##Notebook 9: End to End Demo

This notebook demonstrates MeridianIQ as an end-to-end contract intelligence system.

Unlike earlier notebooks, this notebook does not focus on experimentation or component-level development. Instead, it simulates a real user workflow where a TXT contract is uploaded, processed, analyzed, scored, and converted into a business-readable report.

The goal is to prove that MeridianIQ can operate as a complete AI pipeline, moving from raw contract text to clause detection, evidence retrieval, risk scoring, and executive report generation.

##Loading Core Libraries

In [1]:
import os
import json
import joblib
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer
from getpass import getpass
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 300)

##Importing Reusable MeridianIQ Modules

This step imports the reusable Python modules created from earlier notebook logic.

Instead of repeating large blocks of preprocessing, clause detection, retrieval, risk scoring, and report generation code, MeridianIQ now imports reusable functions from the src folder.

In [2]:
from text_processing import clean_contract_text, chunk_contract_text
from clause_detection import predict_clauses
from evidence_retrieval import retrieve_evidence
from risk_engine import *
from report_generator import build_report_payload, generate_rule_based_report, save_report_payload
from llm_orchestrator import build_gemini_prompt, generate_gemini_report


##Loading Saved Models and Risk Configuration

This step loads the trained TF-IDF vectorizer, trained clause detector, and MeridianIQ risk configuration.

In [3]:
tfidf = joblib.load("tfidf_vectorizer.pkl")
clause_detector = joblib.load("baseline_clause_detector.pkl")

risk_config = pd.read_csv("clause_risk_config.csv")

mvp_config = risk_config[risk_config["is_mvp_clause"] == True].copy()

tfidf, clause_detector, mvp_config.shape

(TfidfVectorizer(max_df=0.95, max_features=20000, min_df=2, ngram_range=(1, 2)),
 OneVsRestClassifier(estimator=LogisticRegression(class_weight='balanced',
                                                  max_iter=1000,
                                                  solver='liblinear')),
 (23, 11))

##Uploading a Raw TXT Contract

This step allows the user to upload a contract text file directly inside Google Colab.

Earlier notebooks worked with CUAD files already inside the dataset folder.
Here, MeridianIQ accepts a user-selected TXT file and runs the full pipeline on that uploaded contract.

In [4]:
from google.colab import files

uploaded = files.upload()


Saving AgapeAtpCorp_20191202_10-KA_EX-10.1_11911128_EX-10.1_Supply Agreement.txt to AgapeAtpCorp_20191202_10-KA_EX-10.1_11911128_EX-10.1_Supply Agreement.txt


##Reading the Uploaded Contract

In [5]:
uploaded_filename = list(uploaded.keys())[0]

with open(uploaded_filename, "r", encoding="utf-8", errors="ignore") as f:
    raw_text = f.read()

print("Uploaded file:", uploaded_filename)
print("Characters:", len(raw_text))
print(raw_text[:1000])

Uploaded file: AgapeAtpCorp_20191202_10-KA_EX-10.1_11911128_EX-10.1_Supply Agreement.txt
Characters: 20693
ODM - SUPPLY AGREEMENT BETWEEN: ORGANIC PREPARATIONS INC. 2nd Floor, Transpacific Haus Lini Highway, Port Vila. Vanuatu "the Manufacturer" -- AND -- AGAPE ATP INTERNATIONAL HOLDING LIMITED Unit 05, 4F, Energy Plaza No. 92, Granville Road Tsim Sha Tsui East Kowloon, Hong Kong "the Customer"

Source: AGAPE ATP CORP, 10-K/A, 12/2/2019





ODM SUPPLY AGREEMENT THIS AGREEMENT is made on the 15t h day of January 2018. BETWEEN: ORGANIC PREPARATIONS INC. 2nd Floor, Transpacific Haus Lini Highway, Port Vila. Vanuatu ('the Manufacturer') of one part AND: AGAPE ATP INTERNATIONAL HOLDING LIMITED Unit 05, 4F, Energy Plaza No. 92, Granville Road Tsim Sha Tsui East Kowloon, Hong Kong ('the Customer') of the other part. RECITALS a. The Manufacturer wishes to appoint the Customer to be the sole and exclusive agent for the promotion, sales, marketing distribution and administration of the Products

##Applying all the Necessary Operations before Report Generation

In [6]:
clean_text = clean_contract_text(raw_text)
chunks_df = chunk_contract_text(clean_text, max_chars=1500)

predicted_fingerprint, detected_clauses_df = predict_clauses(
    clean_text=clean_text,
    tfidf=tfidf,
    clause_detector=clause_detector,
    mvp_config=mvp_config,
    filename=uploaded_filename
)

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

evidence_df = retrieve_evidence(
    chunks_df=chunks_df,
    detected_clauses_df=detected_clauses_df,
    mvp_config=mvp_config,
    embedding_model=embedding_model,
    filename=uploaded_filename,
    top_k=2
)

risk_score, risk_drivers = score_contract(
    predicted_fingerprint.iloc[0],
    mvp_config
)

risk_band = assign_risk_band(risk_score)
risk_driver_table = build_risk_driver_table(risk_drivers, recommendation_map)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

##Generating the Gemini Executive Report

In [7]:
report_payload = build_report_payload(
    filename=uploaded_filename,
    risk_score=risk_score,
    risk_band=risk_band,
    detected_clauses_df=detected_clauses_df,
    risk_driver_table=risk_driver_table,
    evidence_df=evidence_df
)

rule_based_report = generate_rule_based_report(report_payload)
print(rule_based_report[:4000])

print("=" * 70)
print("MeridianIQ Processing Complete")
print("=" * 70)

print(f"Uploaded Contract        : {uploaded_filename}")
print(f"Contract Chunks          : {len(chunks_df)}")
print(f"Detected MVP Clauses     : {len(detected_clauses_df[detected_clauses_df['detected']])}")
print(f"Triggered Risk Drivers   : {len(risk_driver_table)}")
print(f"Overall Risk Score       : {risk_score}/100")
print(f"Overall Risk Band        : {risk_band}")
print("Executive Report         : Generated")
print("Status                   : SUCCESS")

# MeridianIQ End-to-End Contract Review Report

**Generated At:** 2026-06-26 17:02:35 UTC
**Contract File:** AgapeAtpCorp_20191202_10-KA_EX-10.1_11911128_EX-10.1_Supply Agreement.txt

## 1. Overall Risk Assessment

**Risk Score:** 53/100
**Risk Band:** High Risk
**Detected MVP Clauses:** 17
**Triggered Risk Drivers:** 3

## 2. Detected Clauses

- Non-Compete
- Exclusivity
- No-Solicit Of Employees
- Termination For Convenience
- Change Of Control
- Anti-Assignment
- Revenue/Profit Sharing
- Minimum Commitment
- Ip Ownership Assignment
- License Grant
- Irrevocable Or Perpetual License
- Post-Termination Services
- Audit Rights
- Cap On Liability
- Liquidated Damages
- Warranty Duration
- Insurance

## 3. Key Risk Drivers

- **Exclusivity** (Critical, Competition Risk): Creates exclusive dealing obligations that may limit business flexibility.
  - Recommended Action: Review exclusivity obligations and assess whether they restrict future business opportunities.
- **No-Solicit Of Employee

##Final Pipeline Output

This notebook demonstrates the complete MeridianIQ workflow:

Raw TXT Contract
→ Text Cleaning
→ Paragraph-Aware Chunking
→ Clause Detection
→ Semantic Evidence Retrieval
→ Risk Scoring
→ Risk Driver Generation
→ Report Payload Creation
→ Rule-Based Report
→ Gemini Executive Report

##Notebook Summary

This notebook completes the MeridianIQ project by connecting all previous components into one real-time demo.

It demonstrates:

* User-uploaded TXT contract ingestion
* Reusable source-code modules
* Saved model inference
* Clause prediction
* Evidence retrieval
* Risk scoring
* Business recommendation generation
* Rule-based reporting
* Gemini-powered executive reporting


The most important achievement of this notebook is that MeridianIQ becomes a usable workflow rather than a collection of separate experiments.

It proves the final system can take an unseen contract, process it through the full intelligence pipeline, and generate a business-readable contract risk report with supporting evidence.